# Model 1 v2 – 04 Train + in-source evaluation
**Purpose:** Train X-ray vs non-X-ray models on the v2 dataset (more X-ray variety + hard negatives):
MobileNetV3-Large (same architecture as v1) and DeiT-Small; compare with v1 on the v2 test set.
**Inputs:** outputs/model1_v2/manifest_split_v2.csv, cache_224_v2.npz, outputs/model1/mobilenetv3_model1_best.pt (v1)
**Outputs:** outputs/model1_v2/{name}_model1v2_best.pt, preds_*.csv, info_*.json
**Selection:** best epoch = lowest validation loss (accuracy saturates near 100%, loss still shows calibration).

In [3]:
# ===== SETTINGS =====
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Research')
OUT_DIR    = DRIVE_ROOT / 'outputs' / 'model1_v2'
MANIFEST   = OUT_DIR / 'manifest_split_v2.csv'
CACHE      = OUT_DIR / 'cache_224_v2.npz'
V1_MODEL   = DRIVE_ROOT / 'outputs' / 'model1' / 'mobilenetv3_model1_best.pt'

SEED, IMG_SIZE, BATCH_SIZE = 42, 224, 64
EPOCHS, LR, PATIENCE = 10, 1e-4, 3
P_LOWRES, P_JPEG = 0.3, 0.3
MODELS = {'mobilenetv3': 'torchvision', 'deit_small': 'deit_small_patch16_224.fb_in1k'}

In [5]:
!pip install -q timm
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, random, timm, time, os, io, json
device = 'cuda'
df = pd.read_csv(MANIFEST)
cache = np.load(CACHE, allow_pickle=True)
X = cache['images']
assert (cache['paths'][df.cache_idx] == df.path).all(), 'Manifest and cache do not match!'
splits = {s: df[df.split == s].reset_index(drop=True) for s in ['train', 'val', 'test']}
print({s: len(f) for s, f in splits.items()}, '| cache', X.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'train': 6016, 'val': 1258, 'test': 1263} | cache (8685, 224, 224)


In [6]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

def degrade(img):
    if random.random() < P_LOWRES:
        s = random.randint(64, 160)
        img = img.resize((s, s), Image.BILINEAR).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    if random.random() < P_JPEG:
        buf = io.BytesIO(); img.save(buf, format='JPEG', quality=random.randint(30, 90))
        img = Image.open(buf).convert('L')
    return img

def make_loaders(mean, std):
    to_model = [T.ToTensor(), T.Lambda(lambda t: t.repeat(3, 1, 1)), T.Normalize(mean, std)]
    train_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
                          T.RandomHorizontalFlip(), T.RandomRotation(20),
                          T.ColorJitter(brightness=0.2, contrast=0.2), *to_model])
    eval_tf = T.Compose(to_model)
    class DS(Dataset):
        def __init__(self, f, train): self.idx, self.y, self.train = f.cache_idx.values, f.y.values, train
        def __len__(self): return len(self.idx)
        def __getitem__(self, i):
            img = Image.fromarray(X[self.idx[i]])
            return (train_tf(degrade(img)) if self.train else eval_tf(img)), self.y[i]
    return {s: DataLoader(DS(splits[s], s == 'train'), batch_size=BATCH_SIZE, shuffle=(s == 'train'),
                          num_workers=2, pin_memory=True) for s in splits}

In [7]:
import torch.nn as nn, torch.nn.functional as F
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from sklearn.metrics import roc_auc_score

def build(name):
    if MODELS[name] == 'torchvision':
        m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
        return m, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    m = timm.create_model(MODELS[name], pretrained=True, num_classes=2)
    return m, list(m.pretrained_cfg['mean']), list(m.pretrained_cfg['std'])

def metrics(p, y):
    pred = (p >= 0.5).astype(int)
    return {'acc': (pred == y).mean(), 'auc': roc_auc_score(y, p),
            'FAR': ((pred == 1) & (y == 0)).sum() / (y == 0).sum(),     # non-X-ray accepted
            'FRR': ((pred == 0) & (y == 1)).sum() / (y == 1).sum()}     # X-ray rejected

counts = splits['train'].y.value_counts().sort_index().values
w = (counts.sum() / (2 * counts)) ** 0.5; CLASS_W = torch.tensor(w / w.mean(), dtype=torch.float32)

def save_preds(name, split, p):
    out = splits[split][['path', 'source', 'label', 'y']].copy(); out['p_xray'] = p
    out.to_csv(OUT_DIR / f'preds_{name}_{split}.csv', index=False)

def train_and_evaluate(name):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    model, mean, std = build(name); model = model.to(device)
    loaders = make_loaders(mean, std)
    crit = nn.CrossEntropyLoss(weight=CLASS_W.to(device))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda'); path = OUT_DIR / f'{name}_model1v2_best.pt'

    @torch.no_grad()
    def predict(loader):
        model.eval(); ps = []
        for xb, _ in loader:
            with torch.amp.autocast('cuda'):
                ps.append(F.softmax(model(xb.to(device)).float(), 1)[:, 1].cpu())
        return torch.cat(ps).numpy()

    best, best_ep, wait = 1e9, 0, 0
    for ep in range(1, EPOCHS + 1):
        t0 = time.time(); model.train(); tot = 0.0
        for xb, yb in loaders['train']:
            xb, yb = xb.to(device), yb.to(device); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item() * len(yb)
        sched.step()
        p = predict(loaders['val']); y = splits['val'].y.values
        vloss = F.binary_cross_entropy(torch.tensor(p).clamp(1e-6, 1 - 1e-6), torch.tensor(y, dtype=torch.float32)).item()
        m = metrics(p, y)
        print(f'Epoch {ep:2d} | train loss {tot/len(splits["train"]):.4f} | val loss {vloss:.4f} | val acc {m["acc"]:.4f} '
              f'| FAR {m["FAR"]:.3f} | FRR {m["FRR"]:.3f} | {time.time()-t0:.0f}s')
        if vloss < best:
            best, best_ep, wait = vloss, ep, 0
            torch.save({'state_dict': model.state_dict(), 'name': name, 'timm_name': MODELS[name],
                        'mean': mean, 'std': std, 'epoch': ep}, path); print('   ✓ saved best model')
        else:
            wait += 1
            if wait >= PATIENCE: print('Early stopping.'); break

    model.load_state_dict(torch.load(path, map_location=device)['state_dict'])
    for s in ['val', 'test']: save_preds(name, s, predict(loaders[s]))
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device); model.eval()
    with torch.no_grad():
        for _ in range(10): model(x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(100): model(x)
        torch.cuda.synchronize()
    info = {'model': name, 'best_epoch': best_ep, 'params_M': round(sum(q.numel() for q in model.parameters()) / 1e6, 1),
            'size_mb': round(os.path.getsize(path) / 1e6, 1), 'ms_per_img': round((time.time() - t0) * 10, 1)}
    json.dump(info, open(OUT_DIR / f'info_{name}.json', 'w')); print('Done:', info)
    del model; torch.cuda.empty_cache()

In [8]:
train_and_evaluate('mobilenetv3')

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 50.7MB/s]


Epoch  1 | train loss 0.1158 | val loss 0.1879 | val acc 0.9332 | FAR 0.168 | FRR 0.000 | 31s
   ✓ saved best model
Epoch  2 | train loss 0.0114 | val loss 0.0207 | val acc 0.9944 | FAR 0.014 | FRR 0.000 | 19s
   ✓ saved best model
Epoch  3 | train loss 0.0090 | val loss 0.0127 | val acc 0.9944 | FAR 0.014 | FRR 0.000 | 21s
   ✓ saved best model
Epoch  4 | train loss 0.0054 | val loss 0.0066 | val acc 0.9984 | FAR 0.004 | FRR 0.000 | 21s
   ✓ saved best model
Epoch  5 | train loss 0.0045 | val loss 0.0037 | val acc 0.9992 | FAR 0.002 | FRR 0.000 | 20s
   ✓ saved best model
Epoch  6 | train loss 0.0021 | val loss 0.0041 | val acc 0.9992 | FAR 0.002 | FRR 0.000 | 20s
Epoch  7 | train loss 0.0044 | val loss 0.0019 | val acc 0.9992 | FAR 0.002 | FRR 0.000 | 20s
   ✓ saved best model
Epoch  8 | train loss 0.0016 | val loss 0.0011 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 20s
   ✓ saved best model
Epoch  9 | train loss 0.0042 | val loss 0.0009 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 20s


In [9]:
train_and_evaluate('deit_small')

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch  1 | train loss 0.0525 | val loss 0.0045 | val acc 0.9992 | FAR 0.000 | FRR 0.001 | 24s
   ✓ saved best model
Epoch  2 | train loss 0.0150 | val loss 0.0063 | val acc 0.9984 | FAR 0.004 | FRR 0.000 | 23s
Epoch  3 | train loss 0.0094 | val loss 0.0039 | val acc 0.9984 | FAR 0.000 | FRR 0.003 | 22s
   ✓ saved best model
Epoch  4 | train loss 0.0036 | val loss 0.0015 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 23s
   ✓ saved best model
Epoch  5 | train loss 0.0025 | val loss 0.0004 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 23s
   ✓ saved best model
Epoch  6 | train loss 0.0004 | val loss 0.0004 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 23s
   ✓ saved best model
Epoch  7 | train loss 0.0018 | val loss 0.0003 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 23s
   ✓ saved best model
Epoch  8 | train loss 0.0004 | val loss 0.0002 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 24s
   ✓ saved best model
Epoch  9 | train loss 0.0007 | val loss 0.0003 | val acc 1.0000 | FAR 0.000 | FRR 0.000 | 23s


In [10]:
# v1 model on the v2 test set (same preprocessing: 224 gray, ImageNet normalisation)
v1 = mobilenet_v3_large(); v1.classifier[3] = nn.Linear(v1.classifier[3].in_features, 2)
v1.load_state_dict(torch.load(V1_MODEL, map_location=device)); v1.eval().to(device)
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
with torch.no_grad():
    imgs = X[splits['test'].cache_idx.values]
    p_v1 = np.concatenate([torch.softmax(v1(norm(torch.from_numpy(imgs[k:k+64]).float().div(255).unsqueeze(1)
                           .repeat(1, 3, 1, 1)).to(device)), 1)[:, 1].cpu().numpy() for k in range(0, len(imgs), 64)])
save_preds('v1_mobilenetv3', 'test', p_v1)

y = splits['test'].y.values
P = {'v1 MobileNetV3': p_v1,
     'v2 MobileNetV3': pd.read_csv(OUT_DIR / 'preds_mobilenetv3_test.csv').p_xray.values,
     'v2 DeiT-Small':  pd.read_csv(OUT_DIR / 'preds_deit_small_test.csv').p_xray.values}
P['v2 Ensemble'] = (P['v2 MobileNetV3'] + P['v2 DeiT-Small']) / 2
print(pd.DataFrame({k: metrics(v, y) for k, v in P.items()}).T.round(4).to_string())

print('\nErrors per source (test):')
t = splits['test'][['source', 'label']].copy()
for k, v in P.items(): t[k] = ((v >= 0.5).astype(int) != y).astype(int)
err = t.groupby(['label', 'source'])[list(P)].sum(); err.insert(0, 'n', t.groupby(['label', 'source']).size())
print(err[err[list(P)].sum(1) > 0].to_string())

                   acc     auc    FAR     FRR
v1 MobileNetV3  0.9747  0.9986  0.012  0.0340
v2 MobileNetV3  0.9992  1.0000  0.000  0.0013
v2 DeiT-Small   1.0000  1.0000  0.000  0.0000
v2 Ensemble     1.0000  1.0000  0.000  0.0000

Errors per source (test):
                                 n  v1 MobileNetV3  v2 MobileNetV3  v2 DeiT-Small  v2 Ensemble
label    source                                                                               
NON_XRAY new_OCT_OCTMNIST       45               6               0              0            0
X_RAY    m2_Foot_and_ankle      12               2               0              0            0
         m2_Hip_to_knee         35               1               1              0            0
         m2_OTHER               70              16               0              0            0
         m2_Shoulder_to_elbow   75               1               0              0            0
         m2_Wrist_and_hand     204               6               0            

## Summary – in-source test (v2 test set, 1,263 images)
| Model | Acc | FAR | FRR |
|---|---|---|---|
| v1 MobileNetV3 | 97.5% | 1.2% | 3.4% |
| v2 MobileNetV3 | 99.9% | 0% | 0.1% |
| v2 DeiT-Small | 100% | 0% | 0% |
| v2 Ensemble | 100% | 0% | 0% |
v1 errors: 16/70 OTHER (chest/spine/skull) rejected, 6/45 OCT accepted, plus some foot/wrist/shoulder.
v2 fixes these. Caution: same-source test → real generalisation checked in the unseen-source test (notebook 05).
v2 MobileNetV3: 17 MB, 6.3 ms/img; DeiT-Small: 87 MB, 5.5 ms/img (T4).